# GHZ and Bell states

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/tutorials/ghz-bell-state.ipynb)

Entangle three qubits so that they always agree. This is the hello world of quantum computing, and the clearest demonstration that entanglement is real.

Nothing in this notebook costs anything or needs an account until the last section. The result shown is read from the public certificate for the run that actually produced it.

Full write-up: [GHZ and Bell states](https://zksf.org/blog/quantum-entanglement-ghz-bell-state/)


## The idea

A GHZ state puts three qubits into a superposition of all-zeros and all-ones, with nothing in between. One Hadamard creates the superposition on the first qubit, and a chain of CNOTs fans it out so the other two follow.

Measure it and you get `000` or `111`, roughly half the time each. What matters is what you never see: `001`, `010`, or any other mixed string. The three qubits are not three separate coins. They are one linked object.


## The circuit


In [ ]:
!pip install -q qiskit


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 3)
qc.h(0)              # superposition on the first qubit
qc.cx(0, 1)          # fan it out
qc.cx(1, 2)
qc.measure([0, 1, 2], [0, 1, 2])

print(qc.draw(output='text'))


## What happened when this ran

Every gate here is Clifford, so the router sent this to the Stim stabilizer engine, which simulates such circuits **exactly** at any scale. An exact result has no approximation error to report: the only scatter is shot noise.

The cell below reads the real certificate for that run. No account, no cost.


In [ ]:
import requests

CERT = "d904e28af0b441e0"
c = requests.get(f"https://api.zksf.org/certify/{CERT}/json", timeout=30).json()

print(f"engine   : {c['engine']}")
print(f"method   : {c['method']}")
print(f"shots    : {c['shots']}")
if c.get('expectation') is not None:
    print(f"energy   : {c['expectation']:.6f}")
for bits, n in (c.get('top_outcomes') or []):
    print(f"  {bits}  {n}")
print(f"accuracy : {c.get('error_bound', 'exact, shot noise only')}")
print(f"verify   : {c['verify_url']}")


## Reading the result

531 and 469 out of 1,000 is textbook shot noise around the ideal 500/500, the ordinary statistical wobble of a fair coin. The result that matters is what is absent. Not a single shot landed on a mixed string.

That certificate is public. Anyone can open the verify link, or check it programmatically without an account:

```
pip install zcc-verify
zcc-verify d904e28af0b441e0
```


## Run it yourself

Optional, and this part does cost. Circuits this small are a fraction of a cent, and `estimate()` prices any job for free before you commit to it. Get a token from [app.zksf.org](https://app.zksf.org).


In [ ]:
!pip install -q qsim-sdk


In [ ]:
import getpass
import qsim_sdk

client = qsim_sdk.Client(token=getpass.getpass("ZKSF API token: "))

est = client.estimate(qc, shots=1000)
print('engine:', est['engine'], '| cost: $', est['predicted_cost_usd'])


In [ ]:
job = client.run(qc, shots=1000)

print(job['result'].get('counts'))
print(job['result'].get('expectation'))
print(job['result']['error_info'])


## Next

- [The full article](https://zksf.org/blog/quantum-entanglement-ghz-bell-state/), with the maths and the background
- [All tutorials](https://zksf.org/blog/) and the [glossary](https://zksf.org/glossary-of-essential-quantum-computing-terms-for-beginners/)
- [How the accuracy statements work](https://zksf.org/quantum-computing-certification/), and [the paper](https://doi.org/10.5281/zenodo.21851381)
- [Quickstart notebook](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/quickstart.ipynb): four certified runs, including one on real quantum hardware
